In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from geopy.point import Point
from geopy.distance import distance
from sklearn.feature_selection import chi2
from sklearn.feature_selection import SelectKBest
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE
from sklearn.feature_selection import SelectFromModel
import plotly.express as px
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

In [ ]:
df = pd.read_csv("NSQD_data_R_import.csv")

In [ ]:
df = df.iloc[1:]

In [ ]:
df.head(10)

In [ ]:
data_types = df.dtypes.tolist()
print(data_types)

In [ ]:
col = len(df.columns)
rows = len(df)
print(col)
print(rows)

In [ ]:
#several lab QC results columns from testing. Removing
col_q = df.filter(like='Q').columns.tolist()
len(col_q)
df = df.drop(df.filter(like='Q').columns, axis = 1)

In [ ]:
col = len(df.columns)
print(col)

In [ ]:
#create index of columns
qc_col_index = list(enumerate(df.columns))
print(qc_col_index)

In [ ]:
#convert object dtypes to string
df[df.select_dtypes(include=['object']).columns] = df.select_dtypes(include=['object']).astype('string')
df.dtypes

In [ ]:
#getting a count of values that contain '<'.
count_less_than = df.applymap(lambda x: '<' in str(x)).sum().sum()
print(count_less_than)


In [ ]:
#Values that contain '<' are non detect lab results. changing to 0.
df = df.applymap(lambda x: 0 if '<' in str(x) else x)

In [ ]:
#getting a count of values that contain '<'.
count_less_than = df.applymap(lambda x: '<' in str(x)).sum().sum()
print(count_less_than)

In [ ]:
#get a count of values that contain '>'.
count_greater_than = df.applymap(lambda x: '>' in str(x)).sum().sum()
print(count_greater_than)

In [ ]:
#days since last rain count
days_count = df["Days since last rain"].astype(str).str.contains('>8hrs').sum()
print(days_count)

In [ ]:
#BOD, Fecal, Strepococcus and TC that contain '>' are underdiluted. Those results will be removed, days since last rain containing '>8hrs' will be changed to 10.
cols_with_gt = df.columns[df.apply(lambda col: col.astype(str).str.contains('>').any())].tolist()
gt_count = len(cols_with_gt)
print(gt_count)
print(cols_with_gt)

In [ ]:
df['Days since last rain'] = df['Days since last rain'].apply(lambda x: 10 if str(x) == '>8hrs' else x)
days_count = df["Days since last rain"].astype(str).str.contains('>8hrs').sum()
print(days_count)

In [ ]:
len(df)

In [ ]:
#removing the missod dilution results.
df = df[~df.astype(str).apply(lambda row: row.str.contains(">")).any(axis=1)]
count_greater_than = df.applymap(lambda x: '>' in str(x)).sum().sum()
print(count_greater_than)

In [ ]:
n_count = df.select_dtypes(include=['string']).apply(lambda x: x.str.contains('\n')).any(axis=1).sum()
print(n_count)

In [ ]:
#SAved at this break point to save time on reload.
df.to_pickle('current_clean_BP1.pkl')

In [ ]:
#import if needed.
df = pd.read_pickle("current_clean_BP.pkl")

In [ ]:
missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]
print(missing_counts)

In [ ]:
miss_percent_2 = df.isna().sum() / len(df) * 100
miss_percent_2 = miss_percent_2[miss_percent_2 > 0]
print(miss_percent_2)

In [ ]:
#calculating the runoff coef for missing values were available.
df["Runoff_Vol_Coef_for_Site"] = np.where(
    df["Runoff_Vol_Coef_for_Site"].isna() & df["Runoff_(in)"].notna() & df["Precipitation_Depth_(in)"].notna() & (df["Precipitation_Depth_(in)"] != 0),
    df["Runoff_(in)"] / df["Precipitation_Depth_(in)"],
    df["Runoff_Vol_Coef_for_Site"]
)

In [ ]:
miss_percent_2 = df.isna().sum() / len(df) * 100
miss_percent_2 = miss_percent_2[miss_percent_2 > 0]
print(miss_percent_2)

In [ ]:
#calculate the RV where available
df["Calculated_Rv"] = np.where(
    df["Calculated_Rv"].isna() & df["Percent_Impervious"].notna(),
    df["Percent_Impervious"] * 0.9 + 0.05,
    df["Calculated_Rv"]
)

In [ ]:
miss_percent_2 = df.isna().sum() / len(df) * 100
miss_percent_3 = miss_percent_2[miss_percent_2 > 0]
print(miss_percent_3)

In [ ]:
na_percent = df.isna().mean() * 100
cols_to_keep = na_percent[na_percent < 73].index.tolist()
df = df[cols_to_keep]

In [ ]:
miss_percent_2 = df.isna().sum() / len(df) * 100
miss_percent_3 = miss_percent_2[miss_percent_2 > 0]
print(miss_percent_2)

In [ ]:
missing_values = df.isnull().sum()
missing_percent = missing_values / len(df) * 100
missing_percent2 = missing_percent[missing_percent > 0]
print(missing_percent2)

In [ ]:
df = df.dropna(subset=["Runoff_(in)"])

In [ ]:
df = df.dropna(subset=["pH"])

In [ ]:
df = df.dropna(subset=["Percent_Impervious"])

In [ ]:
df = df.dropna(subset=["Calculated_Curve_Number"])

In [ ]:
missing_values = df.isnull().sum()
missing_percent = missing_values / len(df) * 100
missing_percent2 = missing_percent[missing_percent > 0]
print(missing_percent2)

In [ ]:
len(df)

In [ ]:
len(df.columns)

# <font color="red">Do not use import - see next break point</font>

In [ ]:
#SAved at this break point to save time on reload.
df.to_pickle('current_clean_BP.pkl')

In [ ]:
#import if needed.
df = pd.read_pickle("current_clean_BP.pkl")

In [ ]:
df.head()

In [ ]:
df['Latitude'].head()

In [ ]:
df['Latitude'] = df['Latitude'].str.replace('_', ' ')
df['Longitude'] = df['Longitude'].str.replace('_', ' ')
df['Latitude'].head()

In [ ]:
df['Latitude'] = df['Latitude'] + 'N'
df['Longitude'] = df['Longitude'] + 'W'

In [ ]:
df['Longitude'].head()

In [ ]:
missing_values = df['Longitude'].isnull().sum()
print(missing_values)

## <font color=green>following code was used to trouble shoot regex.</font>

In [ ]:
dms_pattern = r'^(\d+)\s+(\d+)\s+(\d+)([NSEW])$'

def is_valid_dms(val):
  if pd.isna(val):
    return False
  return bool(re.match(dms_pattern, str(val).strip()))

In [ ]:
df['Latitude'] = df['Latitude'].astype('string')
df['Longitude'] = df['Longitude'].astype('string')

In [ ]:
results_df = df[['Latitude', 'Longitude']].copy()
results_df['Latitude_valid'] = df['Latitude'].apply(is_valid_dms)
results_df['Longitude_valid'] = df['Longitude'].apply(is_valid_dms)

In [ ]:
results_df.head()

In [ ]:
false_count_lat = (results_df['Latitude_valid'] == False).sum()
print(false_count_lat)

In [ ]:
invalid_lat = results_df[results_df['Latitude_valid'] ==False]
print(invalid_lat)

In [ ]:
mask = df['Longitude'].str.contains('\xa0', regex = False)
print(df[mask])

In [ ]:
extra_white = df['Longitude'].str.contains(r'[^\x20-\x7E]', regex = True)
print(extra_white.sum())

In [ ]:
print(repr(df.loc[6568, 'Latitude']))

In [ ]:
print(repr(df.loc[6568, 'Longitude']))

In [ ]:
print(len(df.loc[6568, 'Latitude']))

In [ ]:
lat_invalid = invalid_lat.loc[8683, 'Longitude']
for c in lat_invalid:
  print(ord(c))


In [ ]:
print(chr(78))

# <font color=green>Use this import with current_clean_BP.pkl</font>

In [ ]:
#import if needed.
df = pd.read_pickle("current_clean_BP.pkl")

In [ ]:
def dms_to_dd(dms_str):
  match = re.match(r'^(\d+)\s+(\d+)\s+(\d+(?:\.\d+)?)([NSEW])$', dms_str.strip())
  if not match:
    return None
  deg = float(match.group(1))
  min = float(match.group(2))
  sec = float(match.group(3))
  direction = match.group(4).upper()
  dd = deg + min / 60 + sec / 3600
  if direction in ['S', 'W']:
    dd *= -1
  return dd

df['Latitude'] = df['Latitude'].apply(dms_to_dd)
df['Longitude'] = df['Longitude'].apply(dms_to_dd)

In [ ]:
print(dms_to_dd('122 11 35.29N'))

In [ ]:
df['Latitude'].head()

In [ ]:
#calculating distance from omaha, location for each sample location. Pretty close to middle, why not.
#approximately middle of omaha
omaha_coords = (41.2565, -95.9345)

def distance_from_omaha(row):
  point = (row['Latitude'], row['Longitude'])
  return distance(omaha_coords, point).km

# Remove rows with NaN in Latitude or Longitude before applying the function
df.dropna(subset=['Latitude', 'Longitude'], inplace=True)

df['Distance'] = df.apply(distance_from_omaha, axis = 1)
df['Distance'].head()

In [ ]:
object_cols = df.select_dtypes(include=['object']).columns.tolist()
print(object_cols)

In [ ]:
cols_to_drop = object_cols[:8]
df = df.drop(cols_to_drop, axis = 1)

In [ ]:
cols_to_drop = ['Controls', 'Start_Date', 'Start_Time','End_Date', 'End_Time', 'Comments']
df.drop(columns=cols_to_drop, inplace=True)

In [ ]:
df[['Days since last rain', 'Age_of_Development']] = df[['Days since last rain', 'Age_of_Development']].astype(float)

In [ ]:
cols_to_convert = object_cols[8:]
df[cols_to_convert] = df[cols_to_convert].astype(float)

In [ ]:
object_cols1 = df.select_dtypes(include=['object']).columns.tolist()
print(object_cols1)

In [ ]:
object_cols1.append('EPA_Rain_Zone')
print(object_cols1)

In [ ]:
col_dtype_df = pd.DataFrame({
    'column_name': df.columns,
    'dtype': [df[col].dtype for col in df.columns]
})
pd.set_option('display.max_rows', len(df))
print(col_dtype_df)

In [ ]:
df[object_cols1] = df[object_cols1].astype(str)

In [ ]:
from sklearn.preprocessing import OneHotEncoder
encoder = OneHotEncoder(sparse_output=False)
encoded_array = encoder.fit_transform(df[object_cols1])
encoded_columns = encoder.get_feature_names_out(object_cols1)
df_encoded = pd.DataFrame(encoded_array, columns=encoded_columns, index = df.index)
display(df_encoded.head())

In [ ]:
df_final = pd.concat([df.drop(object_cols1, axis = 1), df_encoded], axis = 1)

In [ ]:
df_final = df_final.drop('ID_V4.02', axis = 1)

In [ ]:
df_final['Nitrogen_Nitrate (mg/L as N) '].head()

In [ ]:
df_final.head()

In [ ]:
#Run some feature selection methods. Target variables are TSS (mg/L), Nitrogen_Nitrate (mg/L as N), and Phosphorous Total as P (mg/L). Going to run multiple feature selection methods for each target variable.

col_tss = 'TSS (mg/L)'
col_no3 = 'Nitrogen_Nitrate (mg/L as N) '
col_phos = 'Phosphorous Total as P (mg/L)'

cols_tss = [c for c in df_final.columns if c != col_tss]
cols_no3 = [c for c in df_final.columns if c != col_no3]
cols_phos = [c for c in df_final.columns if c != col_phos]

new_order_tss = cols_tss + [col_tss]
new_order_no3 = cols_no3 + [col_no3]
new_order_phos = cols_phos + [col_phos]

df_TSS = df_final[new_order_tss]
df_NO3 = df_final[new_order_no3]
df_Phos = df_final[new_order_phos]

In [ ]:
col_dtype_df_TSS = df_TSS.dtypes.reset_index().rename(columns={"index": "Column", 0: "Dtype"})
print(col_dtype_df_TSS)

In [ ]:
X_TSS = df_TSS.iloc[:, 0:147]
y_TSS = df_TSS.iloc[:, -1]
X_NO3 = df_NO3.iloc[:, 0:147]
y_NO3 = df_NO3.iloc[:, -1]
X_Phos = df_Phos.iloc[:, 0:147]
y_Phos = df_Phos.iloc[:, -1]

In [ ]:
X_NO3.head()

In [ ]:
y_NO3.head()

# <font color=green>TSS Feature Selection</font>

In [ ]:
from sklearn.feature_selection import f_regression

# Use f_regression for feature selection with a continuous target variable
TSS_best_features = SelectKBest(score_func=f_regression, k=50)
fit_TSS = TSS_best_features.fit(X_TSS, y_TSS)

In [ ]:
df_scores = pd.DataFrame(fit_TSS.scores_)
df_columns = pd.DataFrame(X_TSS.columns)

f_scores = pd.concat([df_columns, df_scores], axis = 1)
f_scores.columns = ['Features', 'Scores']

print(f_scores)

In [ ]:
print(f_scores.nlargest(50, 'Scores'))

In [ ]:
scores = TSS_best_features.scores_
feature_names = X_TSS.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('F-score')
plt.xlabel('Top 50 Features')
plt.title('Top 50 Feature TSS F-scores (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('TSS_F_scores.png')

In [ ]:
rf_tss = RandomForestRegressor()
rf_tss.fit(X_TSS, y_TSS)

feature_importance_tss = rf_tss.feature_importances_
feature_importance_tss_df = pd.Series(feature_importance_tss, index=X_TSS.columns).sort_values(ascending=False)
print(feature_importance_tss_df)

In [ ]:
scores = feature_importance_tss
feature_names = X_TSS.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('feature_imoportance_')
plt.xlabel('Top 50 Features')
plt.title('Top 50 TSS feature_importance_ (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('TSS_feature_importance.png')

In [ ]:
#recursive Feature Elimination

selector = RFE(rf_tss, n_features_to_select=25)  # Select top 25 features
selector_TSS = selector.fit(X_TSS, y_TSS)

selected_features_TSS = X_TSS.columns[selector_TSS.support_]
print(selected_features_TSS)

In [ ]:
tss_ranking = selector_TSS.ranking_
feature_names = X_TSS.columns  # Your feature names

tss_ranking_df = pd.DataFrame({'feature': feature_names, 'ranking': tss_ranking})
tss_ranking_df.sort_values('ranking', inplace=True)

plt.barh(tss_ranking_df['feature'], tss_ranking_df['ranking'])
plt.xlabel('RFE Ranking (1 = selected)')
plt.title('TSS Feature Ranking by RFE')
plt.show()
plt.savefig('TSS_RFE_ranking.png')

# <font color=green>NO3 Feature Selection</font>

In [ ]:
# Use f_regression for feature selection with a continuous target variable
NO3_best_features = SelectKBest(score_func=f_regression, k=50)
fit_NO3 = NO3_best_features.fit(X_NO3, y_NO3)

In [ ]:
scores = NO3_best_features.scores_
feature_names = X_NO3.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('F-score')
plt.xlabel('Top 50 Features')
plt.title('Top 50 Feature NO3 F-scores (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('NO3_F_scores.png')

In [ ]:
df_scores = pd.DataFrame(fit_NO3.scores_)
df_columns = pd.DataFrame(X_NO3.columns)

f_scores = pd.concat([df_columns, df_scores], axis = 1)
f_scores.columns = ['Features', 'Scores']

print(f_scores.nlargest(50, 'Scores'))

In [ ]:
rf_NO3 = RandomForestRegressor()
rf_NO3.fit(X_NO3, y_NO3)

feature_importance_NO3 = rf_NO3.feature_importances_
feature_importance_NO3_df = pd.Series(feature_importance_NO3, index=X_NO3.columns).sort_values(ascending=False)
print(feature_importance_NO3_df)

In [ ]:
scores = feature_importance_NO3
feature_names = X_NO3.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('feature_imoportance_')
plt.xlabel('Top 50 Features')
plt.title('Top 50 NO3 feature_importance_ (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('NO3_feature_importance.png')

In [ ]:
#recursive Feature Elimination

selector = RFE(rf_NO3, n_features_to_select=25)  # Select top 25 features
selector_NO3 = selector.fit(X_NO3, y_NO3)

selected_features_NO3 = X_NO3.columns[selector_NO3.support_]
print(selected_features_NO3)

# <font color=green>Phos Feature Selection</font>

In [ ]:
# Use f_regression for feature selection with a continuous target variable
Phos_best_features = SelectKBest(score_func=f_regression, k=50)
fit_Phos = Phos_best_features.fit(X_Phos, y_Phos)

In [ ]:
df_scores = pd.DataFrame(fit_Phos.scores_)
df_columns = pd.DataFrame(X_Phos.columns)

f_scores = pd.concat([df_columns, df_scores], axis = 1)
f_scores.columns = ['Features', 'Scores']

print(f_scores.nlargest(50, 'Scores'))

In [ ]:
scores = Phos_best_features.scores_
feature_names = X_Phos.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('F-score')
plt.xlabel('Top 50 Features')
plt.title('Top 50 Feature Phos F-scores (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('Phos_F_scores.png')

In [ ]:
rf_Phos = RandomForestRegressor()
rf_Phos.fit(X_Phos, y_Phos)

feature_importance_Phos = rf_Phos.feature_importances_
feature_importance_Phos_df = pd.Series(feature_importance_Phos, index=X_Phos.columns).sort_values(ascending=False)
print(feature_importance_Phos_df)

In [ ]:
scores = feature_importance_Phos
feature_names = X_Phos.columns  # Your feature names

# Sort scores and feature names in descending order
sorted_indices = np.argsort(scores)[::-1]
sorted_scores = scores[sorted_indices]
sorted_names = feature_names[sorted_indices]

# Select top 50
top_n = 50
top_scores = sorted_scores[:top_n]
top_names = sorted_names[:top_n]

# Plot
plt.figure(figsize=(14, 6))
plt.plot(top_scores, marker='o')
plt.xticks(ticks=np.arange(top_n), labels=top_names, rotation=90)
plt.ylabel('feature_imoportance_')
plt.xlabel('Top 50 Features')
plt.title('Top 50 Phos feature_importance_ (sorted)')
plt.tight_layout()
plt.grid(True)
plt.show()
plt.savefig('Phos_feature_importance.png')

In [ ]:
#recursive Feature Elimination

selector = RFE(rf_Phos, n_features_to_select=25)  # Select top 25 features
selector_Phos = selector.fit(X_Phos, y_Phos)

selected_features_Phos = X_Phos.columns[selector_Phos.support_]
print(selected_features_Phos)

# <font color=green>Sort and take top 25 features for f-score</font>

In [ ]:

TSS_sorted_indices = np.argsort(TSS_best_features.scores_)[::-1]
TSS_top_25 = TSS_sorted_indices[:25]
TSS_top_features = X_TSS.columns[TSS_top_25]

X_TSS_top25 = X_TSS[TSS_top_features]


In [ ]:
print(X_TSS_top25.shape[1])

In [ ]:
NO3_sorted_indices = np.argsort(NO3_best_features.scores_)[::-1]
NO3_top_25 = NO3_sorted_indices[:25]
NO3_top_features = X_NO3.columns[NO3_top_25]

X_NO3_top25 = X_NO3[NO3_top_features]

In [ ]:
print(X_NO3_top25.shape[1])

In [ ]:
Phos_sorted_indices = np.argsort(Phos_best_features.scores_)[::-1]
Phos_top_25 = Phos_sorted_indices[:25]
Phos_top_features = X_Phos.columns[Phos_top_25]

X_Phos_top25 = X_Phos[Phos_top_features]

In [ ]:
print(X_Phos_top25.shape[1])

In [ ]:
with open('f_score_X_TSS_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_TSS_top25, 'y': y_TSS}, f)

In [ ]:
with open('f_score_X_NO3_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_NO3_top25, 'y': y_NO3}, f)

In [ ]:
with open('f_score_X_Phos_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_Phos_top25, 'y': y_Phos}, f)

# <font color=green>Sort and take top 25 features for RF feature importance</font>

In [ ]:
importances = rf_tss.feature_importances_
feat_imp = pd.Series(importances, index = X_TSS.columns)
X_TSS_top25_RF = feat_imp.sort_values(ascending=False)[:25].index.tolist()

X_TSS_top25_rf = X_TSS[X_TSS_top25_RF]

In [ ]:
X_TSS_top25_rf.head()

In [ ]:
print(X_TSS_top25_rf.shape[1])

In [ ]:
importances_NO3 = rf_NO3.feature_importances_
feat_imp_NO3 = pd.Series(importances_NO3, index = X_NO3.columns)
X_NO3_top25_RF = feat_imp_NO3.sort_values(ascending=False)[:25].index.tolist()

X_NO3_top25_rf = X_NO3[X_NO3_top25_RF]

In [ ]:
X_NO3_top25_rf.head()

In [ ]:
importances_Phos = rf_Phos.feature_importances_
feat_imp_Phos = pd.Series(importances_Phos, index = X_Phos.columns)
X_Phos_top25_RF = feat_imp_Phos.sort_values(ascending=False)[:25].index.tolist()

X_Phos_top25_rf = X_Phos[X_Phos_top25_RF]

In [ ]:
X_Phos_top25_rf.head()

In [ ]:
with open('rf_X_TSS_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_TSS_top25_rf, 'y': y_TSS}, f)

In [ ]:
with open('rf_X_NO3_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_NO3_top25_rf, 'y': y_NO3}, f)

In [ ]:
with open('rf_X_Phos_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_Phos_top25_rf, 'y': y_Phos}, f)

# <font color=green>Sort and take top 25 features for RFE</font>

In [ ]:
TSS_top25 = list(selected_features_TSS)
X_TSS_top25_RFE = X_TSS[TSS_top25]

In [ ]:
X_TSS_top25_RFE.head()

In [ ]:
NO3_top25 = list(selected_features_NO3)
X_NO3_top25_RFE = X_NO3[NO3_top25]

In [ ]:
X_NO3_top25_RFE.head()

In [ ]:
Phos_top25 = list(selected_features_Phos)
X_Phos_top25_RFE = X_Phos[Phos_top25]

In [ ]:
with open('RFE_X_TSS_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_TSS_top25_RFE, 'y': y_TSS}, f)

In [ ]:
with open('REF_X_NO3_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_NO3_top25_RFE, 'y': y_NO3}, f)

In [ ]:
with open('REF_X_Phos_top25_and_y.pkl', 'wb') as f:
    pickle.dump({'X': X_Phos_top25_RFE, 'y': y_Phos}, f)